# Hospital Feature Store - ML Input Pipeline

## Muc tieu: Du doan `inpatient_occupancy_rate` tuan ke tiep

### Kien truc
```
BigQuery (5 tables)
  -> SQL Feature Engineering (lag/roll/join)
fs_hospital_weekly (BQ)    <- Feature Store snapshot
  -> Temporal split
train / val / test         <- 2020-2022 / 2023Q1-Q3 / 2023Q4-2024
  -> XGBoost baseline
ml_artifacts/              <- parquet + models + preprocessor
```

### Features (~41)
| Group | Count | Source |
|---|---|---|
| Lag & Rolling (occ/icu/covid/flu/ed) | 20 | fact |
| Calendar | 8 | dim_date |
| Hospital static | 4 | dim_hospital |
| Demographics | 8 | dim_population |
| Geography | 3 | dim_geography |

### Targets
| Target | Type |
|---|---|
| `target_occupancy_next_week` | FLOAT regression |
| `target_high_strain` | BOOL classification (>85%) |

In [1]:
# Install
!pip install google-cloud-bigquery db-dtypes pyarrow pandas scikit-learn xgboost -q
print('Libraries ready')

Libraries ready


In [2]:
# Configuration
PROJECT_ID  = "project-8e2366a6-d3cc-40ee-9de"
DATASET_ID  = "hospital_dwh_dev"
FS_DATASET  = "hospital_feature_store" # <- feature store dataset (will be created)
FS_TABLE    = "fs_hospital_weekly"     # <- main feature store table

TRAIN_END   = "2022-12-31"
VAL_END     = "2023-09-30"
RANDOM_SEED = 42

print(f"Source : {PROJECT_ID}.{DATASET_ID}")
print(f"Feature Store : {PROJECT_ID}.{FS_DATASET}.{FS_TABLE}")
print(f"Train  : 2020-08-01 -> {TRAIN_END}")
print(f"Val    : to {VAL_END}")
print(f"Test   : to 2024-05-10")

Source : project-8e2366a6-d3cc-40ee-9de.hospital_dwh_dev
Feature Store : project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.fs_hospital_weekly
Train  : 2020-08-01 -> 2022-12-31
Val    : to 2023-09-30
Test   : to 2024-05-10


In [3]:
# Auth & BigQuery client
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np, os, pickle, warnings
warnings.filterwarnings("ignore")

bq = bigquery.Client(project=PROJECT_ID)
fs_ds = bigquery.Dataset(f"{PROJECT_ID}.{FS_DATASET}")
fs_ds.location = "asia-southeast1"
bq.create_dataset(fs_ds, exists_ok=True)
print(f"Connected to {PROJECT_ID}")
print(f"Feature Store dataset [{FS_DATASET}] ready")

Connected to project-8e2366a6-d3cc-40ee-9de
Feature Store dataset [hospital_feature_store] ready


In [6]:
# Feature Engineering SQL
# Mục đích: Tạo bảng Feature Store phẳng bằng cách kết nối 5 bảng BigQuery nguồn
# để sinh ra hàng loạt đặc trưng chuỗi thời gian, lịch trình, công suất bệnh viện và nhân khẩu học.
# truy vấn SQL tối ưu thành 4 bảng tạm thời (CTE)
# base: Trích xuất dữ liệu thô từ bảng thực tế (fact) kết hợp các chiều ngày tháng từ dim_date.
# lagged: Sử dụng hàm cửa sổ (Window functions) để tính toán độ trễ quá khứ và trung bình trượt.
# hosp: Trích xuất đặc trưng tĩnh từ bảng dim_hospital dựa trên snapshot mới nhất.
# pop: Tổng hợp chỉ số dân cư và địa lý từ dim_population và dim_geography.
def build_feature_sql(project, dataset):
    return f"""
WITH base AS (
  SELECT
    f.hospital_id,
    f.collection_week                                          AS report_date,
    f.state,
    COALESCE(f.inpatient_beds_capacity,  0)                    AS inpatient_capacity,
    COALESCE(f.inpatient_beds_used,      0)                    AS inpatient_used,
    COALESCE(f.inpatient_occupancy_rate, 0)                    AS occupancy_rate,
    COALESCE(f.icu_beds_capacity,        0)                    AS icu_capacity,
    COALESCE(f.icu_beds_used,            0)                    AS icu_used,
    COALESCE(f.icu_occupancy_rate,       0)                    AS icu_occupancy_rate,
    COALESCE(f.covid_hospitalized_patients,    0)              AS covid_patients,
    COALESCE(f.covid_icu_patients,             0)              AS covid_icu,
    COALESCE(f.influenza_hospitalized_patients,0)              AS flu_patients,
    COALESCE(f.influenza_icu_patients,         0)              AS flu_icu,
    COALESCE(f.ed_visits_total,                0)              AS ed_visits,
    COALESCE(f.covid_admissions_adult,         0)              AS covid_admit_adult,
    COALESCE(f.covid_admissions_pediatric,     0)              AS covid_admit_peds,
    COALESCE(f.pediatric_beds_capacity,        0)              AS peds_capacity,
    COALESCE(f.pediatric_beds_used,            0)              AS peds_used,
    CAST(COALESCE(f.is_corrected, FALSE) AS INT64)             AS is_corrected,
    d.week_number,
    d.month,
    d.quarter,
    d.year,
    d.season,
    CAST(d.is_holiday AS INT64)         AS is_holiday,
    CAST(d.is_flu_season AS INT64)      AS is_flu_season,
    d.disease_season,
    d.healthcare_risk_level,
    d.week_of_month

  FROM `{project}.{dataset}.fact_hospital_utilization` f

  LEFT JOIN `{project}.{dataset}.dim_date` d
    ON f.collection_week = d.full_date

  WHERE f.inpatient_beds_capacity > 0
    AND f.collection_week BETWEEN '2020-08-01' AND '2024-05-10'
),

lagged AS (

  SELECT *,

    LAG(occupancy_rate,1)  OVER w AS occ_lag1,
    LAG(occupancy_rate,2)  OVER w AS occ_lag2,
    LAG(occupancy_rate,3)  OVER w AS occ_lag3,
    LAG(occupancy_rate,4)  OVER w AS occ_lag4,
    LAG(occupancy_rate,8)  OVER w AS occ_lag8,
    LAG(occupancy_rate,12) OVER w AS occ_lag12,

    AVG(occupancy_rate) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS occ_roll4,

    AVG(occupancy_rate) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 7 PRECEDING AND CURRENT ROW
    ) AS occ_roll8,

    LAG(icu_occupancy_rate,1) OVER w AS icu_lag1,
    LAG(icu_occupancy_rate,4) OVER w AS icu_lag4,

    AVG(icu_occupancy_rate) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS icu_roll4,

    LAG(covid_patients,1) OVER w AS covid_lag1,
    LAG(covid_patients,2) OVER w AS covid_lag2,
    LAG(covid_patients,4) OVER w AS covid_lag4,

    AVG(covid_patients) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS covid_roll4,

    LAG(flu_patients,1) OVER w AS flu_lag1,
    LAG(flu_patients,4) OVER w AS flu_lag4,

    AVG(flu_patients) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS flu_roll4,

    LAG(ed_visits,1) OVER w AS ed_lag1,

    AVG(ed_visits) OVER (
      PARTITION BY hospital_id
      ORDER BY report_date
      ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS ed_roll4,

    SAFE_DIVIDE(
      occupancy_rate - LAG(occupancy_rate,1) OVER w,
      NULLIF(LAG(occupancy_rate,1) OVER w, 0)
    ) AS occ_wow_pct_change,

    LEAD(occupancy_rate,1) OVER w AS target_occupancy_next_week

  FROM base

  WINDOW w AS (
    PARTITION BY hospital_id
    ORDER BY report_date
  )
),

-- dim_hospital schema:
-- hospital_id, hospital_name, hospital_type,
-- certified_bed_count, county_fips, zip_code

hosp AS (

  SELECT
    hospital_id,
    hospital_name,
    zip_code,
    county_fips,
    hospital_type,
    COALESCE(certified_bed_count, 0) AS certified_bed_count

  FROM `{project}.{dataset}.dim_hospital`

  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY hospital_id
    ORDER BY hospital_id
  ) = 1
),

pop AS (

  SELECT
    p.county_fips,
    p.total_population,
    p.pop_65_plus_pct,
    p.population_density,
    p.median_household_income,
    p.poverty_rate_pct,
    p.uninsured_rate_pct,
    p.physicians_per_100k,
    p.nurses_per_100k,
    p.rucc_code,
    p.metro_nonmetro_flag,
    g.county_name,
    g.hrr_region,
    g.area_km2

  FROM `{project}.{dataset}.dim_population` p

  LEFT JOIN `{project}.{dataset}.dim_geography` g
    USING (county_fips)

  WHERE p.census_year = (
    SELECT MAX(census_year)
    FROM `{project}.{dataset}.dim_population`
  )
)

SELECT

  -- Keys
  l.hospital_id,
  l.report_date,
  l.state,

  -- Hospital info
  h.hospital_name,
  h.zip_code,

  -- Current utilization
  l.inpatient_capacity,
  l.inpatient_used,
  l.occupancy_rate,
  l.icu_capacity,
  l.icu_used,
  l.icu_occupancy_rate,
  l.covid_patients,
  l.covid_icu,
  l.flu_patients,
  l.flu_icu,
  l.ed_visits,
  l.covid_admit_adult,
  l.covid_admit_peds,
  l.peds_capacity,
  l.peds_used,
  l.is_corrected,

  -- Lag & rolling features
  l.occ_lag1,
  l.occ_lag2,
  l.occ_lag3,
  l.occ_lag4,
  l.occ_lag8,
  l.occ_lag12,
  l.occ_roll4,
  l.occ_roll8,

  l.icu_lag1,
  l.icu_lag4,
  l.icu_roll4,

  l.covid_lag1,
  l.covid_lag2,
  l.covid_lag4,
  l.covid_roll4,

  l.flu_lag1,
  l.flu_lag4,
  l.flu_roll4,

  l.ed_lag1,
  l.ed_roll4,

  l.occ_wow_pct_change,

  -- Calendar features
  l.week_number,
  l.month,
  l.quarter,
  l.year,
  l.season,
  l.is_holiday,
  l.is_flu_season,
  l.disease_season,
  l.healthcare_risk_level,
  l.week_of_month,

  -- Hospital static
  h.hospital_type,
  h.certified_bed_count,
  h.county_fips,

  -- Population / geography
  p.total_population,
  p.pop_65_plus_pct,
  p.population_density,
  p.median_household_income,
  p.poverty_rate_pct,
  p.uninsured_rate_pct,
  p.physicians_per_100k,
  p.nurses_per_100k,
  p.rucc_code,
  p.metro_nonmetro_flag,
  p.county_name,
  p.hrr_region,
  p.area_km2,

  -- Targets
  l.target_occupancy_next_week,

  CAST(
    l.target_occupancy_next_week > 0.85
    AS INT64
  ) AS target_high_strain,

  CURRENT_TIMESTAMP() AS feature_computed_at

FROM lagged l

LEFT JOIN hosp h
  USING (hospital_id)

LEFT JOIN pop p
  ON h.county_fips = p.county_fips

WHERE l.target_occupancy_next_week IS NOT NULL
  AND l.occ_lag4 IS NOT NULL

ORDER BY hospital_id, report_date
"""

FEATURE_SQL = build_feature_sql(PROJECT_ID, DATASET_ID)

print("Feature Engineering SQL ready")
print("dim_hospital fields used:")
print("hospital_id, hospital_name, hospital_type,")
print("certified_bed_count, county_fips, zip_code")
print("state field sourced from fact_hospital_utilization")
print("Lag features: occupancy, icu, covid, flu, ed")
print("Targets: target_occupancy_next_week + target_high_strain")

Feature Engineering SQL ready
dim_hospital fields used:
hospital_id, hospital_name, hospital_type,
certified_bed_count, county_fips, zip_code
state field sourced from fact_hospital_utilization
Lag features: occupancy, icu, covid, flu, ed
Targets: target_occupancy_next_week + target_high_strain


In [7]:
# Create Feature Store table in BigQuery
print("Running Feature Engineering SQL on BigQuery...")
print("(2-5 minutes depending on data size)")

dest_table = f"{PROJECT_ID}.{FS_DATASET}.{FS_TABLE}"
job_cfg = bigquery.QueryJobConfig(
    destination=dest_table,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    allow_large_results=True,
    use_query_cache=False,
)
job = bq.query(FEATURE_SQL, job_config=job_cfg)
job.result()

tbl = bq.get_table(dest_table)
print(f"Feature Store saved!")
print(f"  Table  : {dest_table}")
print(f"  Rows   : {tbl.num_rows:,}")
print(f"  Columns: {len(tbl.schema)}")
print(f"  Size   : {tbl.num_bytes/1024/1024:.1f} MB")

Running Feature Engineering SQL on BigQuery...
(2-5 minutes depending on data size)
Feature Store saved!
  Table  : project-8e2366a6-d3cc-40ee-9de.hospital_feature_store.fs_hospital_weekly
  Rows   : 18,181
  Columns: 71
  Size   : 10.2 MB


In [8]:
# Load into pandas and data quality check
from IPython.display import display

print("Loading feature store...")
fs_df = bq.query(
    f"SELECT * FROM `{PROJECT_ID}.{FS_DATASET}.{FS_TABLE}` ORDER BY hospital_id, report_date"
).to_dataframe()
print(f"Loaded {len(fs_df):,} rows x {fs_df.shape[1]} columns")

print("\n--- DATA QUALITY REPORT ---")
print(f"Unique hospitals : {fs_df['hospital_id'].nunique():,}")
print(f"Date range       : {fs_df['report_date'].min()} -> {fs_df['report_date'].max()}")
print(f"Weeks covered    : {fs_df['report_date'].nunique()}")
print(f"States           : {fs_df['state'].nunique()}")
print(f"\nTarget distribution:")
print(f"  Occupancy mean  : {fs_df['target_occupancy_next_week'].mean():.1%}")
print(f"  High strain pct : {fs_df['target_high_strain'].mean():.1%}")
print(f"  Occupancy > 1.0 : {(fs_df['target_occupancy_next_week'] > 1.0).sum():,} (will be clipped)")

missing = (fs_df.isnull().sum() / len(fs_df) * 100).sort_values(ascending=False)
missing_nonzero = missing[missing > 0].head(10)
print("\nMissing values:")
if len(missing_nonzero) > 0:
    display(missing_nonzero.rename("Missing %").reset_index())
else:
    print("  None!")

numeric_cols = fs_df.select_dtypes(include=['number']).columns.tolist()
corr = fs_df[numeric_cols].corr()["target_occupancy_next_week"].abs()
corr = corr.drop(["target_occupancy_next_week","target_high_strain"], errors="ignore")
print("\nTop 10 correlations with target:")
display(corr.sort_values(ascending=False).head(10).rename("Abs Corr").reset_index())

Loading feature store...
Loaded 18,181 rows x 71 columns

--- DATA QUALITY REPORT ---
Unique hospitals : 3,730
Date range       : 2020-10-25 -> 2024-04-14
Weeks covered    : 182
States           : 56

Target distribution:
  Occupancy mean  : 67.3%
  High strain pct : 26.3%
  Occupancy > 1.0 : 80 (will be clipped)

Missing values:


,index,Missing %
0,occ_lag12,90.319564
1,occ_lag8,65.337440
2,population_density,3.212145
3,total_population,3.212145
4,county_name,3.212145
5,pop_65_plus_pct,3.212145
6,poverty_rate_pct,3.212145
7,rucc_code,3.212145
8,metro_nonmetro_flag,3.212145
9,physicians_per_100k,3.212145



Top 10 correlations with target:


,index,Abs Corr
0,occupancy_rate,0.864610
1,occ_roll4,0.860238
2,occ_roll8,0.833392
3,occ_lag1,0.818275
4,occ_lag2,0.774875
5,occ_lag3,0.735206
6,occ_lag4,0.686087
7,occ_lag8,0.613877
8,occ_lag12,0.560684
9,inpatient_used,0.416193


In [9]:
# Temporal Train / Val / Test Split
# use temporal split to prevent data leakage

fs_df["report_date"] = pd.to_datetime(fs_df["report_date"])

train_df = fs_df[fs_df["report_date"] <= TRAIN_END].copy()
val_df   = fs_df[(fs_df["report_date"] > TRAIN_END) & (fs_df["report_date"] <= VAL_END)].copy()
test_df  = fs_df[fs_df["report_date"] > VAL_END].copy()

print("Temporal Split complete:")
print(f"{'Split':<8} {'Rows':>8}  {'Hospitals':>10}  {'Date range':<37}  {'High strain':>11}")
print("-" * 82)
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    dr = f"{df['report_date'].min().date()} -> {df['report_date'].max().date()}"
    hs = df['target_high_strain'].mean()
    print(f"{name:<8} {len(df):>8,}  {df['hospital_id'].nunique():>10,}  {dr:<37}  {hs:>10.1%}")

# Define feature sets
EXCLUDE = [
    "hospital_id","report_date","county_fips","zip_code","county_name","hospital_name","city",
    "feature_computed_at","target_occupancy_next_week","target_high_strain",
    "occupancy_rate",  # current week - available in lags
]
CATEGORICAL = ["state","hospital_type","season","disease_season",
               "healthcare_risk_level","metro_nonmetro_flag","hrr_region"]
FEATURE_COLS    = [c for c in fs_df.columns if c not in EXCLUDE]
NUMERIC_FEATS   = [c for c in FEATURE_COLS if c not in CATEGORICAL]
CAT_FEATS       = [c for c in CATEGORICAL if c in FEATURE_COLS]

print(f"\nFeature columns : {len(FEATURE_COLS)} total")
print(f"  Numeric       : {len(NUMERIC_FEATS)}")
print(f"  Categorical   : {len(CAT_FEATS)}")
print(f"  Categorical   : {CAT_FEATS}")

Temporal Split complete:
Split        Rows   Hospitals  Date range                             High strain
----------------------------------------------------------------------------------
Train       9,553       2,882  2020-10-25 -> 2022-12-25                    27.1%
Val         6,034       2,821  2023-01-01 -> 2023-09-24                    25.4%
Test        2,594       1,625  2023-10-01 -> 2024-04-14                    25.4%

Feature columns : 61 total
  Numeric       : 54
  Categorical   : 7
  Categorical   : ['state', 'hospital_type', 'season', 'disease_season', 'healthcare_risk_level', 'metro_nonmetro_flag', 'hrr_region']


In [10]:
# Preprocessing Pipeline
from sklearn.pipeline      import Pipeline
from sklearn.compose       import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute        import SimpleImputer

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value",
                               unknown_value=-1, encoded_missing_value=-1)),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, NUMERIC_FEATS),
    ("cat", cat_pipe, CAT_FEATS),
], remainder="drop")

# Clip occupancy at 1.0 (some hospitals report >100%)
y_tr_reg  = train_df["target_occupancy_next_week"].clip(0,1).values
y_va_reg  = val_df  ["target_occupancy_next_week"].clip(0,1).values
y_te_reg  = test_df ["target_occupancy_next_week"].clip(0,1).values
y_tr_cls  = train_df["target_high_strain"].values.astype(int)
y_va_cls  = val_df  ["target_high_strain"].values.astype(int)
y_te_cls  = test_df ["target_high_strain"].values.astype(int)

print("Fitting preprocessor on train set...")
X_train = preprocessor.fit_transform(train_df[FEATURE_COLS])
X_val   = preprocessor.transform(val_df[FEATURE_COLS])
X_test  = preprocessor.transform(test_df[FEATURE_COLS])
all_feat_names = NUMERIC_FEATS + CAT_FEATS

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train reg range : {y_tr_reg.min():.3f} -> {y_tr_reg.max():.3f}")
print(f"y_train cls pos   : {y_tr_cls.mean():.1%}")

os.makedirs("/content/ml_artifacts", exist_ok=True)
with open("/content/ml_artifacts/preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)
print("Preprocessor saved -> /content/ml_artifacts/preprocessor.pkl")

Fitting preprocessor on train set...
X_train : (9553, 61)
X_val   : (6034, 61)
X_test  : (2594, 61)
y_train reg range : 0.000 -> 1.000
y_train cls pos   : 27.1%
Preprocessor saved -> /content/ml_artifacts/preprocessor.pkl


In [11]:
# Save Parquet + upload scaled BQ ML tables
print("Saving Parquet files (raw features before scaling)...")
for name, df in [("train",train_df),("val",val_df),("test",test_df)]:
    path = f"/content/ml_artifacts/{name}_features.parquet"
    save_cols = FEATURE_COLS + ["target_occupancy_next_week","target_high_strain",
                                "hospital_id","report_date"]
    df[save_cols].to_parquet(path, index=False)
    mb = os.path.getsize(path)/1024/1024
    print(f"  {name}_features.parquet : {len(df):,} rows, {mb:.1f} MB")

print("\nUploading scaled arrays to BigQuery ML tables...")
def arrays_to_bq(X, y_reg, y_cls, split_name):
    feat_df = pd.DataFrame(X, columns=all_feat_names)
    feat_df["target_occupancy_next_week"] = y_reg
    feat_df["target_high_strain"]         = y_cls
    feat_df["split"] = split_name
    tbl = f"{PROJECT_ID}.{FS_DATASET}.ml_{split_name}"
    bq.load_table_from_dataframe(
        feat_df, tbl,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    ).result()
    print(f"  ml_{split_name} -> {len(feat_df):,} rows x {feat_df.shape[1]} cols")

arrays_to_bq(X_train, y_tr_reg, y_tr_cls, "train")
arrays_to_bq(X_val,   y_va_reg, y_va_cls, "val")
arrays_to_bq(X_test,  y_te_reg, y_te_cls, "test")

# Feature catalog
feat_meta = pd.DataFrame({
    "feature_name" : all_feat_names,
    "feature_type" : ["numeric"]*len(NUMERIC_FEATS) + ["categorical"]*len(CAT_FEATS),
    "feature_group": [
        ("lag_roll" if any(x in c for x in ["lag","roll","wow"]) else
         "utilization" if any(x in c for x in ["inpatient","icu","covid","flu","ed","peds"]) else
         "calendar" if any(x in c for x in ["week","month","quarter","year","season","holiday","flu_season","disease","risk"]) else
         "demographics" if any(x in c for x in ["pop","income","poverty","uninsured","physicians","nurses","rucc","metro","area"]) else
         "hospital") for c in NUMERIC_FEATS
    ] + ["categorical"]*len(CAT_FEATS)
})
bq.load_table_from_dataframe(
    feat_meta, f"{PROJECT_ID}.{FS_DATASET}.feature_metadata",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()
print(f"  feature_metadata -> {len(feat_meta)} features catalogued")
print("\nAll outputs saved!")

Saving Parquet files (raw features before scaling)...
  train_features.parquet : 9,553 rows, 1.4 MB
  val_features.parquet : 6,034 rows, 1.0 MB
  test_features.parquet : 2,594 rows, 0.5 MB

Uploading scaled arrays to BigQuery ML tables...
  ml_train -> 9,553 rows x 64 cols
  ml_val -> 6,034 rows x 64 cols
  ml_test -> 2,594 rows x 64 cols
  feature_metadata -> 61 features catalogued

All outputs saved!


In [ ]:
# Summary
print('''
+------------------------------------------------------------+
|  FEATURE STORE PIPELINE COMPLETE                          |
+------------------------------------------------------------+
|                                                            |
|  BigQuery tables created:                                 |
|    fs_hospital_weekly    <- full feature snapshot         |
|    ml_train              <- scaled train set              |
|    ml_val                <- scaled validation set         |
|    ml_test               <- scaled test set               |
|    feature_metadata      <- feature catalog               |
|                                                            |
|  Local files (/content/ml_artifacts/):                    |
|    train/val/test_features.parquet  (raw, pre-scale)      |
|    preprocessor.pkl                                       |
|    xgb_reg_baseline.pkl                                   |
|    xgb_cls_baseline.pkl                                   |
|    feature_importance.png                                 |
|                                                            |
+------------------------------------------------------------+         |
|                                                            |
+------------------------------------------------------------+
''')